# 05 - Evaluation

Comprehensive evaluation of trained models.

**Analysis:**
- Classification metrics (accuracy, AUC, log loss)
- Calibration analysis (are predicted 70% bouts winning ~70%?)
- Accuracy by confidence level
- Comparison to baselines (ELO-only, rank-only)
- SHAP feature importance analysis

**Inputs:**
- `features.parquet` from notebook 03
- `winner_model.lgb` from notebook 04
- `kimarite_model.lgb` from notebook 04
- `test_predictions.parquet` from notebook 04

**Outputs:**
- Evaluation metrics and visualizations

In [ ]:
# Environment setup
import sys
import os

INPUT_PATH = '/kaggle/input/sumo-data-04' if os.path.exists('/kaggle/input') else './output'
FEATURES_PATH = '/kaggle/input/sumo-data-03' if os.path.exists('/kaggle/input') else './output'
OUTPUT_PATH = '/kaggle/working' if os.path.exists('/kaggle/working') else './output'

if not os.path.exists('/kaggle/input'):
    sys.path.insert(0, '../src')

print(f"Input path: {INPUT_PATH}")
print(f"Features path: {FEATURES_PATH}")

In [ ]:
!pip install -q lightgbm scikit-learn shap matplotlib

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score, roc_auc_score, log_loss, brier_score_loss,
    confusion_matrix, classification_report, roc_curve
)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## Load Models and Data

In [ ]:
# Load models
winner_model = lgb.Booster(model_file=f"{INPUT_PATH}/winner_model.lgb")
kimarite_model = lgb.Booster(model_file=f"{INPUT_PATH}/kimarite_model.lgb")
le = joblib.load(f"{INPUT_PATH}/kimarite_encoder.joblib")

print("Models loaded successfully")
print(f"Kimarite classes: {le.classes_}")

In [ ]:
# Load test predictions
test_preds = pd.read_parquet(f"{INPUT_PATH}/test_predictions.parquet")
print(f"Loaded {len(test_preds):,} test predictions")
display(test_preds.head())

In [ ]:
# Load full features for detailed analysis
features_df = pd.read_parquet(f"{FEATURES_PATH}/features.parquet")
print(f"Loaded {len(features_df):,} total bouts")

## Winner Model Evaluation

In [ ]:
# Extract targets and predictions
y_true = test_preds['east_won'].astype(int)
y_pred_proba = test_preds['pred_east_win_prob']
y_pred = test_preds['pred_east_win']

In [ ]:
# Core metrics
print("=== Winner Model - Core Metrics ===")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"AUC: {roc_auc_score(y_true, y_pred_proba):.4f}")
print(f"Log Loss: {log_loss(y_true, y_pred_proba):.4f}")
print(f"Brier Score: {brier_score_loss(y_true, y_pred_proba):.4f}")
print(f"\nBaseline (random 50%): Accuracy=0.5000, AUC=0.5000")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(f"                 Pred West  Pred East")
print(f"Actual West     {cm[0, 0]:8d}  {cm[0, 1]:8d}")
print(f"Actual East     {cm[1, 0]:8d}  {cm[1, 1]:8d}")

## Calibration Analysis

Are predicted probabilities well-calibrated? A bout predicted at 70% should win ~70% of the time.

In [ ]:
def compute_calibration(y_true, y_pred_proba, n_bins=10):
    """Compute calibration metrics."""
    bins = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.digitize(y_pred_proba, bins) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)
    
    bin_data = []
    for i in range(n_bins):
        mask = bin_indices == i
        if mask.sum() > 0:
            bin_data.append({
                'bin': f"{bins[i]:.1f}-{bins[i+1]:.1f}",
                'mean_pred': y_pred_proba[mask].mean(),
                'mean_actual': y_true[mask].mean(),
                'count': mask.sum(),
                'error': abs(y_pred_proba[mask].mean() - y_true[mask].mean())
            })
    
    df = pd.DataFrame(bin_data)
    ece = (df['error'] * df['count']).sum() / df['count'].sum()
    
    return df, ece

cal_df, ece = compute_calibration(y_true.values, y_pred_proba.values)
print(f"Expected Calibration Error (ECE): {ece:.4f}")
print(f"\n(ECE < 0.05 is considered well-calibrated)")
print("\nCalibration by prediction bin:")
display(cal_df)

In [ ]:
# Plot calibration curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calibration curve
ax = axes[0]
prob_true, prob_pred = calibration_curve(y_true, y_pred_proba, n_bins=10)
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.plot(prob_pred, prob_true, 's-', markersize=8, label=f'Model (ECE={ece:.3f})')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives (actual)')
ax.set_title('Calibration Curve')
ax.legend()
ax.grid(True, alpha=0.3)

# Prediction distribution
ax = axes[1]
ax.hist(y_pred_proba[y_true == 0], bins=50, alpha=0.5, label='West won', density=True)
ax.hist(y_pred_proba[y_true == 1], bins=50, alpha=0.5, label='East won', density=True)
ax.axvline(x=0.5, color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('Predicted P(East wins)')
ax.set_ylabel('Density')
ax.set_title('Distribution of Predictions')
ax.legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/calibration_plot.png", dpi=150, bbox_inches='tight')
plt.show()

## Accuracy by Confidence Level

In [ ]:
def accuracy_by_confidence(y_true, y_pred_proba):
    """Compute accuracy by prediction confidence."""
    # Confidence = distance from 0.5
    confidence = np.abs(y_pred_proba - 0.5)
    
    bins = [
        (0.00, 0.05, '50-55%'),
        (0.05, 0.10, '55-60%'),
        (0.10, 0.15, '60-65%'),
        (0.15, 0.20, '65-70%'),
        (0.20, 0.25, '70-75%'),
        (0.25, 0.50, '75%+'),
    ]
    
    results = []
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    for low, high, label in bins:
        mask = (confidence >= low) & (confidence < high)
        if mask.sum() > 0:
            acc = accuracy_score(y_true[mask], y_pred[mask])
            results.append({
                'confidence': label,
                'accuracy': acc,
                'count': mask.sum(),
                'pct': mask.sum() / len(y_true) * 100
            })
    
    return pd.DataFrame(results)

conf_df = accuracy_by_confidence(y_true.values, y_pred_proba.values)
print("Accuracy by prediction confidence:")
display(conf_df)

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(10, 5))

x = range(len(conf_df))
ax.bar(x, conf_df['accuracy'], color='steelblue', alpha=0.7)
ax.axhline(y=0.5, color='r', linestyle='--', label='Random baseline')

ax.set_xticks(x)
ax.set_xticklabels(conf_df['confidence'])
ax.set_xlabel('Prediction Confidence Level')
ax.set_ylabel('Accuracy')
ax.set_title('Model Accuracy by Confidence Level')
ax.set_ylim(0.4, 1.0)

# Add count labels
for i, row in conf_df.iterrows():
    ax.annotate(f'n={row["count"]}', (i, row['accuracy'] + 0.02), ha='center', fontsize=9)

ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/accuracy_by_confidence.png", dpi=150, bbox_inches='tight')
plt.show()

## Comparison to Baselines

In [ ]:
# Merge test predictions with features for baseline comparisons
test_with_features = test_preds.merge(
    features_df[['bout_id', 'elo_diff', 'glicko_rating_diff', 'rank_diff', 
                 'east_elo', 'west_elo', 'east_career_win_rate', 'west_career_win_rate']],
    on='bout_id',
    how='left'
)

print(f"Merged data: {len(test_with_features)} rows")

In [ ]:
# Create baseline predictions
def elo_baseline(elo_diff):
    """ELO expected score formula."""
    return 1.0 / (1.0 + 10 ** (-elo_diff / 400.0))

test_with_features['baseline_elo'] = elo_baseline(test_with_features['elo_diff'])
test_with_features['baseline_rank'] = (test_with_features['rank_diff'] > 0).astype(float) * 0.6 + 0.2
test_with_features['baseline_random'] = 0.5

In [ ]:
# Compare metrics
comparison = []

for name, pred_col in [
    ('LightGBM Model', 'pred_east_win_prob'),
    ('ELO Only', 'baseline_elo'),
    ('Rank Only', 'baseline_rank'),
    ('Random (50%)', 'baseline_random'),
]:
    y_pred_baseline = test_with_features[pred_col].fillna(0.5)
    
    comparison.append({
        'Model': name,
        'Accuracy': accuracy_score(y_true, (y_pred_baseline > 0.5).astype(int)),
        'AUC': roc_auc_score(y_true, y_pred_baseline) if name != 'Random (50%)' else 0.5,
        'Log Loss': log_loss(y_true, np.clip(y_pred_baseline, 0.001, 0.999)),
        'Brier Score': brier_score_loss(y_true, y_pred_baseline),
    })

comparison_df = pd.DataFrame(comparison)
print("\n=== Model Comparison ===")
display(comparison_df)

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy comparison
ax = axes[0]
colors = ['steelblue', 'orange', 'green', 'gray']
ax.bar(comparison_df['Model'], comparison_df['Accuracy'], color=colors)
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy Comparison')
ax.set_ylim(0.4, 0.8)
ax.axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=15)

# AUC comparison
ax = axes[1]
ax.bar(comparison_df['Model'], comparison_df['AUC'], color=colors)
ax.set_ylabel('AUC')
ax.set_title('AUC Comparison')
ax.set_ylim(0.4, 0.8)
ax.axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=15)

plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/model_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

## ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Model ROC
fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
auc_model = roc_auc_score(y_true, y_pred_proba)
ax.plot(fpr, tpr, label=f'LightGBM (AUC={auc_model:.3f})', linewidth=2)

# ELO baseline ROC
fpr_elo, tpr_elo, _ = roc_curve(y_true, test_with_features['baseline_elo'])
auc_elo = roc_auc_score(y_true, test_with_features['baseline_elo'])
ax.plot(fpr_elo, tpr_elo, label=f'ELO Only (AUC={auc_elo:.3f})', linewidth=2)

# Random baseline
ax.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.500)')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/roc_curves.png", dpi=150, bbox_inches='tight')
plt.show()

## SHAP Feature Importance

In [ ]:
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    print("SHAP not available - skipping SHAP analysis")
    SHAP_AVAILABLE = False

In [ ]:
if SHAP_AVAILABLE:
    # Load feature columns
    feature_cols = pd.read_csv(f"{INPUT_PATH}/feature_columns.csv")['0'].tolist()
    
    # Prepare test features
    test_features = test_with_features.merge(
        features_df[['bout_id'] + feature_cols],
        on='bout_id',
        how='left'
    )
    
    # Sample for SHAP (computational efficiency)
    sample_size = min(5000, len(test_features))
    test_sample = test_features.sample(sample_size, random_state=42)
    X_sample = test_sample[feature_cols].fillna(0)
    
    print(f"Computing SHAP values for {len(X_sample)} samples...")

In [ ]:
if SHAP_AVAILABLE:
    # Compute SHAP values
    explainer = shap.TreeExplainer(winner_model)
    shap_values = explainer.shap_values(X_sample)
    
    print("SHAP values computed")

In [ ]:
if SHAP_AVAILABLE:
    # Feature importance from SHAP
    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'mean_abs_shap': np.abs(shap_values).mean(axis=0)
    }).sort_values('mean_abs_shap', ascending=False)
    
    print("\nTop 20 Features by SHAP Importance:")
    display(importance_df.head(20))

In [ ]:
if SHAP_AVAILABLE:
    # SHAP summary plot
    plt.figure(figsize=(10, 12))
    shap.summary_plot(shap_values, X_sample, max_display=20, show=False)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_PATH}/shap_summary.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if SHAP_AVAILABLE:
    # Bar plot of importance
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_sample, plot_type='bar', max_display=20, show=False)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_PATH}/shap_importance.png", dpi=150, bbox_inches='tight')
    plt.show()

## LightGBM Feature Importance (Fallback)

In [ ]:
# LightGBM built-in importance
feature_cols = pd.read_csv(f"{INPUT_PATH}/feature_columns.csv")['0'].tolist()

lgb_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance_gain': winner_model.feature_importance(importance_type='gain'),
    'importance_split': winner_model.feature_importance(importance_type='split')
}).sort_values('importance_gain', ascending=False)

print("\nTop 20 Features by LightGBM Gain:")
display(lgb_importance.head(20))

In [ ]:
# Plot LightGBM importance
fig, ax = plt.subplots(figsize=(10, 10))

top20 = lgb_importance.head(20)
ax.barh(range(len(top20)), top20['importance_gain'][::-1])
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['feature'][::-1])
ax.set_xlabel('Importance (Gain)')
ax.set_title('Top 20 Features by LightGBM Importance')

plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/lgb_importance.png", dpi=150, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print("="*60)
print("EVALUATION SUMMARY")
print("="*60)

print(f"\n--- Winner Prediction Model ---")
print(f"Test Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Test AUC: {roc_auc_score(y_true, y_pred_proba):.4f}")
print(f"Test Log Loss: {log_loss(y_true, y_pred_proba):.4f}")
print(f"Expected Calibration Error: {ece:.4f}")

print(f"\n--- Improvement over Baselines ---")
model_acc = accuracy_score(y_true, y_pred)
elo_acc = accuracy_score(y_true, (test_with_features['baseline_elo'] > 0.5).astype(int))
print(f"vs ELO Only: +{(model_acc - elo_acc)*100:.1f}% accuracy")
print(f"vs Random: +{(model_acc - 0.5)*100:.1f}% accuracy")

print(f"\n--- Top 5 Features ---")
for i, row in lgb_importance.head(5).iterrows():
    print(f"  {i+1}. {row['feature']}")

print(f"\n--- Plots Saved ---")
print(f"  {OUTPUT_PATH}/calibration_plot.png")
print(f"  {OUTPUT_PATH}/accuracy_by_confidence.png")
print(f"  {OUTPUT_PATH}/model_comparison.png")
print(f"  {OUTPUT_PATH}/roc_curves.png")
print(f"  {OUTPUT_PATH}/lgb_importance.png")

## Next Steps

Evaluation complete! Ready for:
- `06_narrative_generation.ipynb` - Generate qualitative bout previews